In [ ]:
!pip install pandas scikit-learn kagglehub ipywidgets



In [ ]:
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# import kagglehub

# # Download latest version
# path = kagglehub.competition_download('smart-mcq-solver-challenge')

# print("Path to competition files:", path)

100%|██████████| 404k/404k [00:01<00:00, 361kB/s]

Extracting files...
Path to competition files: /home/shreyas/.cache/kagglehub/competitions/smart-mcq-solver-challenge


In [5]:
train_df = pd.read_csv('/home/shreyas/.cache/kagglehub/competitions/smart-mcq-solver-challenge/train.csv')
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 125.1 KB


In [7]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1

In [24]:
train_df['answer'].value_counts()
490+324

814

In [9]:
train_df['prompt'] = train_df['prompt'].str.lower()
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer: what is martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,what is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,determine the correct option: what is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,select the most accurate option: what is marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,identify the correct statement: what is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
translator = str.maketrans('','',string.punctuation)

In [11]:
train_df['prompt'] = train_df['prompt'].str.translate(translator)
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer what is martin h...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,what is acceleratorbased lightion fusion,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,determine the correct option what is the term ...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,select the most accurate option what is martin...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,identify the correct statement what is the con...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [12]:
train_df["prompt_tokens"] = train_df["prompt"].str.split()
train_df.head()

,id,prompt,A,B,C,D,E,answer,prompt_tokens
0,1,pick the best possible answer what is martin h...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,"[pick, the, best, possible, answer, what, is, ..."
1,2,what is acceleratorbased lightion fusion,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,"[what, is, acceleratorbased, lightion, fusion]"
2,3,determine the correct option what is the term ...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,"[determine, the, correct, option, what, is, th..."
3,4,select the most accurate option what is martin...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,"[select, the, most, accurate, option, what, is..."
4,5,identify the correct statement what is the con...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,"[identify, the, correct, statement, what, is, ..."


## Q2

In [13]:
vocab = set()
for tokens in train_df["prompt_tokens"]:
    vocab.update(tokens)
len(vocab)

859

## Q3

In [14]:
prompt_ID1 = train_df.loc[train_df['id'] == 1, 'prompt_tokens'].iloc[0]
print(prompt_ID1)

['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']


In [15]:
filtered_words = [w for w in prompt_ID1 if w not in ENGLISH_STOP_WORDS]
len(filtered_words)

13

## Q4

In [16]:
combined_text = (
    train_df['prompt']+' '+
    train_df['A']+' '+
    train_df['B']+' '+
    train_df['C']+' '+
    train_df['D']+' '+
    train_df['E']
)


In [17]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_text)

X.shape

(2000, 2807)

## Q5

In [25]:
row_ID = train_df.loc[train_df['id'] == 1].iloc[0]

prompt_ID1 = row_ID['prompt']
opt_a_ID1 = row_ID['A']

prompt_vec = vectorizer.transform([prompt_ID1])
option_a_vec = vectorizer.transform([opt_a_ID1])

print(cosine_similarity(prompt_vec,option_a_vec)[0][0])


0.18475412231731114


## Q6

In [19]:
correct = 0

for _, row in train_df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0, 0]

    predicted = max(sims, key=sims.get)

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / len(train_df)) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 13.05%


## Q9

In [20]:
counts = train_df["answer"].value_counts()
print(counts)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [21]:
top3 = counts.index[:3].tolist()
print(top3)

['B', 'C', 'A']


In [22]:
counts = train_df["answer"].value_counts()

first, second, third = counts.index[:3]

score_map = {
    first: 1.0,
    second: 0.5,
    third: 1/3
}

map3 = train_df["answer"].map(score_map).fillna(0).mean()

print(round(map3, 4))

0.4212


## Q10

In [26]:

def map_at_3(actual, predicted):
    try:
        rank = predicted.index(actual) + 1
        return 1 / rank
    except ValueError:
        return 0.0

scores = []

for _, row in train_df.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])

        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0, 0]

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    scores.append(
        map_at_3(row["answer"], top3)
    )

final_map3 = sum(scores) / len(scores)

print(f"MAP@3 = {final_map3:.4f}")

MAP@3 = 0.2835
